# DL Lab I — Train a Small NN — AI Lab Instructor Notebook

*Deep Learning · Hard*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# DL Lab I — Train a Small NN (NumPy) — Student Lab

Goal: implement a 2-layer MLP + softmax cross-entropy + backprop, then train it on two-moons.

## Section 0 — Data + baseline

We’ll do a clean train/val split and compare against LogisticRegression.

In [ ]:
import math
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

X, y = make_moons(n_samples=2000, noise=0.22, random_state=0)
X = X.astype(float)
y = y.astype(int)

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
check('shapes', X_tr.shape[0] == 1500 and X_va.shape[0] == 500 and X_tr.shape[1] == 2)

baseline = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
baseline.fit(X_tr, y_tr)
print('baseline train acc', float(baseline.score(X_tr, y_tr)))
print('baseline val   acc', float(baseline.score(X_va, y_va)))

**Why this works.** # DL Lab I — Train a Small NN (NumPy) — Instructor Lab

Solutions + short rationales. Focus: stable softmax CE, backprop, and a clean training loop.

## Section 0 — Data + baseline

Rationale: always compare to a strong linear baseline and keep scaling leakage-free.

## Task 2: Implement parameter init and `forward(params, X)` returning `(logits, cache)`.
*Section: MLP forward pass*

## Section 1 — MLP forward pass

We implement a 2-layer MLP: 
- `H = relu(XW1 + b1)`
- `logits = HW2 + b2`

### Task 1.1
Implement parameter init and `forward(params, X)` returning `(logits, cache)`.

In [ ]:
def relu(x):
    return np.maximum(0.0, x)

def init_params(d_in, d_hidden, d_out, seed=0):
    r = np.random.default_rng(seed)
    W1 = r.normal(0.0, math.sqrt(2.0 / d_in), size=(d_in, d_hidden))
    b1 = np.zeros((1, d_hidden))
    W2 = r.normal(0.0, math.sqrt(2.0 / d_hidden), size=(d_hidden, d_out))
    b2 = np.zeros((1, d_out))
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

def forward(params, X):
    Z1 = X @ params['W1'] + params['b1']
    H = relu(Z1)
    logits = H @ params['W2'] + params['b2']
    cache = {'X': X, 'Z1': Z1, 'H': H}
    return logits, cache

params = init_params(d_in=2, d_hidden=32, d_out=2, seed=0)
logits, cache = forward(params, X_tr[:10])
check('logits_shape', logits.shape == (10, 2))

In [ ]:
# Checks
check('logits_shape', logits.shape == (10, 2))

**Why this works.** ## Section 1 — MLP forward

Rationale: keep everything shape-explicit and cache intermediates needed for backward.

## Task 3: Implement numerically stable softmax.
*Section: Softmax + cross-entropy*

## Section 2 — Softmax + cross-entropy

### Task 2.1
Implement numerically stable softmax.

### Task 2.2
Implement cross-entropy loss (mean over batch).

In [ ]:
def softmax(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    expz = np.exp(z)
    return expz / expz.sum(axis=1, keepdims=True)

def cross_entropy(probs, y):
    n = probs.shape[0]
    p = probs[np.arange(n), y]
    return float(-np.mean(np.log(p + 1e-12)))

probs = softmax(logits)
print('loss', cross_entropy(probs, y_tr[:10]))
check('prob_rows_sum1', np.allclose(probs.sum(axis=1), 1.0, atol=1e-6))

In [ ]:
# Checks
check('prob_rows_sum1', np.allclose(probs.sum(axis=1), 1.0, atol=1e-6))

**Why this works.** ## Section 2 — Softmax + Cross-Entropy

Rationale: stable softmax prevents overflow; CE uses only the correct-class probability.

## Task 4: Implement `backward(params, cache, probs, y)` returning grads dict.
*Section: Backprop*

## Section 3 — Backprop

We’ll use the standard simplification: for softmax + CE, `dlogits = (probs - onehot(y)) / n`.

### Task 3.1
Implement `backward(params, cache, probs, y)` returning grads dict.

### Task 3.2
Finite-difference check one parameter entry.

In [ ]:
$f

In [ ]:
# Checks
check('grad_close', abs(fd - an) < 1e-3)

**Why this works.** ## Section 3 — Backprop

Rationale: the gradient of softmax+CE simplifies to `(probs - onehot(y))/n`.

## Task 5: Implement a training loop with:
*Section: Training loop (mini-batch SGD)*

## Section 4 — Training loop (mini-batch SGD)

### Task 4.1
Implement a training loop with:
- minibatches
- weight decay (L2)
- metrics (train/val acc)

HINT: standardize inputs using train-set mean/std.

**Hints**
- standardize inputs using train-set mean/std.

In [ ]:
$$11

In [ ]:
# Checks
check('beats_baseline_val', accuracy(params, Xva, y_va) >= float(baseline.score(X_va, y_va)) - 0.02)

**Why this works.** ## Section 4 — Training loop

Rationale: the training loop is where most bugs hide: data ordering, batching, scaling leakage, and update ordering.

## Task 6: Overfit a tiny batch (e.g., 32 samples) to near-100% accuracy.
*Section: Debugging playbook*

## Section 5 — Debugging playbook

### Task 5.1
Overfit a tiny batch (e.g., 32 samples) to near-100% accuracy.
If you can’t, your gradients/updates are wrong.

In [ ]:
Xtiny = Xtr[:32]
ytiny = y_tr[:32]
p = init_params(d_in=2, d_hidden=64, d_out=2, seed=2)
lr = 0.15
for step in range(300):
    logits, cache = forward(p, Xtiny)
    probs = softmax(logits)
    grads = backward(p, cache, probs, ytiny)
    p['W1'] -= lr * grads['W1']
    p['b1'] -= lr * grads['b1']
    p['W2'] -= lr * grads['W2']
    p['b2'] -= lr * grads['b2']
    if step % 60 == 0:
        acc = float((predict(p, Xtiny) == ytiny).mean())
        loss = cross_entropy(softmax(forward(p, Xtiny)[0]), ytiny)
        print('step', step, 'acc', acc, 'loss', loss)

tiny_acc = float((predict(p, Xtiny) == ytiny).mean())
print('tiny_acc', tiny_acc)
check('overfit_tiny_batch', tiny_acc > 0.95)

In [ ]:
# Checks
check('overfit_tiny_batch', tiny_acc > 0.95)

**Why this works.** ## Section 5 — Debugging: overfit-a-batch

Rationale: if you can’t overfit 32 samples, something is wrong (gradients, updates, loss).